# Étape 2 — Rémi : qualité des données

## Objectif

Vérifier que le jeu de données est suffisamment propre avant de poursuivre l'analyse exploratoire : aperçu, valeurs manquantes, doublons, incohérences et valeurs atypiques.


In [4]:
import pandas as pd

from load_data import load_prepared

df = load_prepared()
df.head()

print("Dimensions :", df.shape)

# 1. Valeurs manquantes : que manque-t-il ?
missing = (
    df.isna().sum()
      .rename("n_manquants")
      .to_frame()
      .assign(pourcentage=lambda x: (100 * x["n_manquants"] / len(df)).round(2))
      .query("n_manquants > 0")
      .sort_values("n_manquants", ascending=False)
)
display(missing)

# 2. Doublons : y a-t-il des lignes répétées ?
duplicates = pd.DataFrame({
    "doublons_complets": [int(df.duplicated().sum())],
    "doublons_user_film": [int(df.duplicated(["user_id", "item_id"]).sum())],
})
display(duplicates)

# 3. Catégories : les valeurs sont-elles cohérentes ?
category_summary = pd.DataFrame({
    "colonne": ["gender", "occupation", "rating"],
    "nunique": [
        int(df["gender"].nunique(dropna=True)),
        int(df["occupation"].nunique(dropna=True)),
        int(df["rating"].nunique(dropna=True)),
    ],
    "valeurs": [
        ", ".join(sorted(df["gender"].dropna().astype(str).unique())),
        ", ".join(sorted(df["occupation"].dropna().astype(str).unique())),
        ", ".join(str(value) for value in sorted(df["rating"].dropna().unique())),
    ],
})
display(category_summary)

display(df["gender"].value_counts(dropna=False).rename_axis("gender").to_frame("effectif"))
display(df["rating"].value_counts(dropna=False).sort_index().rename_axis("rating").to_frame("effectif"))

# 4. Min, max et quantiles : y a-t-il des valeurs impossibles ou surprenantes ?
numeric_columns = ["rating", "age", "timestamp", "release_year"]
numeric_summary = df[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
numeric_summary = numeric_summary[["min", "1%", "25%", "50%", "75%", "99%", "max"]]
display(numeric_summary)

range_checks = pd.DataFrame({
    "controle": [
        "notes hors de l'intervalle [1, 5]",
        "âges hors de l'intervalle plausible [1, 120]",
        "années de sortie hors [1888, année actuelle]",
        "timestamps négatifs",
        "dates converties invalides",
    ],
    "effectif": [
        int((df["rating"].notna() & ~df["rating"].between(1, 5)).sum()),
        int((df["age"].notna() & ~df["age"].between(1, 120)).sum()),
        int((df["release_year"].notna() & ~df["release_year"].between(1888, pd.Timestamp.now().year)).sum()),
        int((df["timestamp"].notna() & (df["timestamp"] < 0)).sum()),
        int(df["datetime"].isna().sum()),
    ],
})
display(range_checks)

# 5. Dates, unités et types : sont-ils cohérents ?
type_summary = df.dtypes.astype(str).rename("type").to_frame()
display(type_summary)

date_summary = pd.DataFrame({
    "indicateur": [
        "timestamp min",
        "timestamp max",
        "date min",
        "date max",
        "années de sortie manquantes",
    ],
    "valeur": [
        df["timestamp"].min(),
        df["timestamp"].max(),
        df["datetime"].min(),
        df["datetime"].max(),
        int(df["release_year"].isna().sum()),
    ],
})
display(date_summary)

print("Conclusion :")
print("- Les valeurs manquantes, les doublons et les catégories ont été vérifiés.")
print("- Les min, max et quantiles permettent de repérer les valeurs impossibles ou atypiques.")
print("- Les dates, les unités et les types sont contrôlés avant les étapes 3 et 4.")
print("- Les valeurs manquantes de release_date et release_year correspondent à des dates absentes du catalogue.")



Dimensions : (100000, 38)


,n_manquants,pourcentage
video_release_date,100000,100.00
imdb_url,13,0.01
release_date,9,0.01
release_year,9,0.01


,doublons_complets,doublons_user_film
0,0,0


,colonne,nunique,valeurs
0,gender,2,"F, M"
1,occupation,21,"administrator, artist, doctor, educator, engin..."
2,rating,5,"1, 2, 3, 4, 5"


,effectif
gender,
M,74260
F,25740


...

,min,1%,25%,50%,75%,99%,max
rating,1.0,1.0,3.0,4.0,4.0,5.0,5.0
age,7.0,15.0,24.0,30.0,40.0,61.0,73.0
timestamp,874724710.0,874825826.0,879448709.5,882826944.0,888259984.0,892911494.0,893286638.0
release_year,1922.0,1939.0,1986.0,1994.0,1996.0,1997.0,1998.0


,controle,effectif
0,"notes hors de l'intervalle [1, 5]",0
1,"âges hors de l'intervalle plausible [1, 120]",0
2,"années de sortie hors [1888, année actuelle]",0
3,timestamps négatifs,0
4,dates converties invalides,0


,type
user_id,int64
item_id,int64
rating,int64
timestamp,int64
datetime,"datetime64[s, UTC]"
date,object
month,str
weekday,str
hour,int32
age,int64


,indicateur,valeur
0,timestamp min,874724710
1,timestamp max,893286638
2,date min,1997-09-20 03:05:10+00:00
3,date max,1998-04-22 23:10:38+00:00
4,années de sortie manquantes,9


Conclusion :
- Les valeurs manquantes, les doublons et les catégories ont été vérifiés.
- Les min, max et quantiles permettent de repérer les valeurs impossibles ou atypiques.
- Les dates, les unités et les types sont contrôlés avant les étapes 3 et 4.
- Les valeurs manquantes de release_date et release_year correspondent à des dates absentes du catalogue.
